<a href="https://colab.research.google.com/github/jonathan-ha1/Problem-Set-3/blob/main/ps3_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [2]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [3]:
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, size=N_CUSTOMERS)

customers = pd.DataFrame({
    "customer_id": np.arange(1, N_CUSTOMERS + 1),
    "region": rng.choice(["West", "East"], size=N_CUSTOMERS, p=[0.7, 0.3]),
    "revenue": np.maximum(REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, N_CUSTOMERS), REV_FLOOR).round(2),
    "signup_date": np.datetime64("2023-01-01") + rng.integers(0, 730, size=N_CUSTOMERS),
})

orders = pd.DataFrame({"customer_id": np.repeat(customers["customer_id"].to_numpy(), n_orders)})
orders.insert(0, "order_id", np.arange(1, len(orders) + 1))

con.register("customers_df", customers)
con.register("orders_df", orders)
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders_df")

print(con.sql("SELECT region, COUNT(*) AS customers, ROUND(AVG(revenue), 2) AS avg_revenue FROM customers GROUP BY region ORDER BY region"))
print(con.sql("SELECT COUNT(*) AS orders, COUNT(DISTINCT customer_id) AS customers_with_orders FROM orders"))

┌─────────┬───────────┬─────────────┐
│ region  │ customers │ avg_revenue │
│ varchar │   int64   │   double    │
├─────────┼───────────┼─────────────┤
│ East    │       944 │      830.29 │
│ West    │      2056 │      806.04 │
└─────────┴───────────┴─────────────┘

┌────────┬───────────────────────┐
│ orders │ customers_with_orders │
│ int64  │         int64         │
├────────┼───────────────────────┤
│  20490 │                  3000 │
└────────┴───────────────────────┘



### Step 1.2: Write both queries (7 pts)

In [4]:
PLAIN_ROWS = "SELECT customer_id, revenue FROM customers WHERE region = 'West'"
JOINED_ROWS = "SELECT c.customer_id, c.revenue FROM customers c JOIN orders o ON o.customer_id = c.customer_id WHERE c.region = 'West'"

avg_plain, n_plain = con.execute(f"SELECT AVG(revenue), COUNT(*) FROM ({PLAIN_ROWS})").fetchone()
avg_joined, n_joined = con.execute(f"SELECT AVG(revenue), COUNT(*) FROM ({JOINED_ROWS})").fetchone()

print(f"West, un-joined: ${avg_plain:,.2f} over {n_plain:,} rows")
print(f"West, joined:    ${avg_joined:,.2f} over {n_joined:,} rows")
print(f"Ratio: {avg_joined / avg_plain:.2f}")

West, un-joined: $806.04 over 2,056 rows
West, joined:    $1,231.19 over 13,866 rows
Ratio: 1.53


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

The un-joined number is the right one. Revenue is a customer-level number, so each customer should only count once, and the un-joined query has one row per customer. The joined query counts each customer once for every order they placed, and since customers with more orders also have higher revenue, they pull the average way up.

### Step 1.3: The rule (10 pts)

In [5]:
def grain_check(con, row_sql, key):
    n_rows, n_keys = con.execute(f"SELECT COUNT(*), COUNT(DISTINCT {key}) FROM ({row_sql})").fetchone()
    if n_rows > n_keys:
        return f"FAN-OUT: {n_rows:,} rows for {n_keys:,} unique {key}"
    return None

print("joined:", grain_check(con, JOINED_ROWS, "customer_id"))
print("plain: ", grain_check(con, PLAIN_ROWS, "customer_id"))

joined: FAN-OUT: 13,866 rows for 2,056 unique customer_id
plain:  None


This check compares how many rows a query returns to how many different customers are in it. If there are more rows than customers, the join repeated people, so any average from it is off. It warns on the joined query and stays quiet on the un-joined one.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [6]:
null_ids = rng.choice(customers["customer_id"].to_numpy(), size=int(0.15 * N_CUSTOMERS), replace=False)
con.register("null_ids", pd.DataFrame({"customer_id": null_ids}))
con.execute("UPDATE customers SET region = NULL WHERE customer_id IN (SELECT customer_id FROM null_ids)")

print(con.sql("SELECT region, COUNT(*) AS customers FROM customers GROUP BY region ORDER BY region"))

┌─────────┬───────────┐
│ region  │ customers │
│ varchar │   int64   │
├─────────┼───────────┤
│ East    │       812 │
│ West    │      1738 │
│ NULL    │       450 │
└─────────┴───────────┘



### Step 2.2: Lose them, then find them (15 pts)

In [7]:
n_ne = con.execute("SELECT COUNT(*) FROM customers WHERE region != 'West'").fetchone()[0]
n_meant = con.execute("SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'").fetchone()[0]

print(f"WHERE region != 'West':                {n_ne:,} rows")
print(f"WHERE region IS DISTINCT FROM 'West':  {n_meant:,} rows")
print(f"Rows dropped by != :                   {n_meant - n_ne:,}")

WHERE region != 'West':                812 rows
WHERE region IS DISTINCT FROM 'West':  1,262 rows
Rows dropped by != :                   450


The != query returned 450 fewer rows, which matches the 450 customers with no region. SQL treats NULL != ‘West’ as unknown instead of true, and WHERE only keeps true rows, so those customers are dropped without an error.

If those missing customers are systematically different (newer, smaller, or more likely to churn), every regional comparison excludes them. As a result, the customer base appears larger and more stable than it actually is, and the problem goes undetected because the query produces no error.

### Step 2.3: Aggregate without collapsing (10 pts)

In [8]:
window_df = con.execute("""
    SELECT customer_id, region, revenue,
           AVG(revenue) OVER (PARTITION BY region) AS region_avg,
           revenue - AVG(revenue) OVER (PARTITION BY region) AS vs_region
    FROM customers
    ORDER BY customer_id
""").df()

print(f"rows in: {N_CUSTOMERS:,}   rows out: {len(window_df):,}")
print(window_df.head(8).round(2).to_string(index=False))

rows in: 3,000   rows out: 3,000
 customer_id region  revenue  region_avg  vs_region
           1   West   959.53      805.00     154.53
           2   East   832.94      832.04       0.90
           3   West   448.80      805.00    -356.20
           4   West   766.48      805.00     -38.52
           5   West   890.58      805.00      85.58
           6   East  1416.89      832.04     584.85
           7   West   781.98      805.00     -23.02
           8   West   473.01      805.00    -331.99


## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [9]:
TRUE_EFFECT = 20.0
n_sim = 5_000

account_size = rng.normal(100, 30, n_sim)
west = (rng.random(n_sim) < 1 / (1 + np.exp(-(account_size - 100) / 15))).astype(int)
revenue_sim = 300 + 5 * account_size + TRUE_EFFECT * west + rng.normal(0, 100, n_sim)

sim = pd.DataFrame({"account_size": account_size, "west": west, "revenue": revenue_sim})

naive = sim.loc[sim.west == 1, "revenue"].mean() - sim.loc[sim.west == 0, "revenue"].mean()
adjusted = smf.ols("revenue ~ west + account_size", data=sim).fit().params["west"]

print(f"truth {TRUE_EFFECT:.2f} | naive {naive:.2f} | adjusted {adjusted:.2f}")

truth 20.00 | naive 208.45 | adjusted 20.97


The program’s true effect is 20 dollars. The naive comparison shows West customers earning about 208.64 dollars more, which makes the program look much more effective than it is. That’s because bigger accounts were more likely to be put in the West, and bigger accounts already earn more. After controlling for account size, the estimate drops to about 21.93 dollars, which is close to the true 20 dollars.

### Step 3.2: The control that ruins it (13 pts)

In [10]:
tickets = rng.poisson(np.maximum(0, 10 + 3 * west + 0.03 * (revenue_sim - 800)))
sim["tickets"] = tickets

with_tickets = smf.ols("revenue ~ west + account_size + tickets", data=sim).fit().params["west"]

print(f"truth {TRUE_EFFECT:.2f} | naive {naive:.2f} | adjusted {adjusted:.2f} | adding tickets {with_tickets:.2f}")

truth 20.00 | naive 208.45 | adjusted 20.97 | adding tickets -32.64


Adding support tickets as a control made the estimate worse, moving it from about 21.93 dollars, which is close to the true 20 dollars, all the way down to −30.62 dollars, which makes the programme look like it hurts revenue. This happens because both the programme and higher revenue lead customers to file more tickets, so when we compare customers with the same number of tickets, the West customers got some of their tickets from the programme and must have lower revenue to end up at the same count. Account size works as a control because it existed before the programme, but tickets don’t, since the programme itself changes how many tickets customers file.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [11]:
import re
import duckdb
import pandas as pd

_SQL_KEYWORDS = {"WHERE", "JOIN", "LEFT", "RIGHT", "INNER", "FULL", "OUTER", "CROSS",
                 "ON", "USING", "GROUP", "ORDER", "HAVING", "LIMIT", "UNION", "AS"}


def _alias_map(sql):
    # Map every alias (o, c) and every bare table name to the real table it points at.
    aliases = {}
    pattern = r"\b(?:FROM|JOIN)\s+([A-Za-z_]\w*)(?:\s+(?:AS\s+)?([A-Za-z_]\w*))?"
    for table, alias in re.findall(pattern, sql, flags=re.IGNORECASE):
        aliases[table.lower()] = table.lower()
        if alias and alias.upper() not in _SQL_KEYWORDS:
            aliases[alias.lower()] = table.lower()
    return aliases


def _columns(con, table):
    # The column names DuckDB has on file for a table; an empty set means it is not a real table.
    rows = con.execute(
        "SELECT column_name FROM information_schema.columns WHERE table_name = ?", [table]
    ).fetchall()
    return {r[0].lower() for r in rows}


def review_query(con, sql):
    warnings = []
    aliases = _alias_map(sql)
    real_tables = {t for t in set(aliases.values()) if _columns(con, t)}
    fanned_out = False

    # CHECK 1: a join that changes the row count.
    # How fan-out is detected without running the full join twice:
    # a join can only multiply rows when the join key repeats on at least one side.
    # So instead of running the join and comparing row counts, each side of every
    # ON a.x = b.y condition is checked on its own table: one GROUP BY on that single
    # table returns the total rows, the number of distinct keys, and the most rows any
    # one key has. If total rows > distinct keys, every matching row on the other side
    # will be repeated, up to max_per_key times. The join itself is never executed.
    for left_alias, left_col, right_alias, right_col in re.findall(
        r"\bON\s+(\w+)\.(\w+)\s*=\s*(\w+)\.(\w+)", sql, flags=re.IGNORECASE
    ):
        sides = [(left_alias, left_col, right_alias), (right_alias, right_col, left_alias)]
        for alias, col, other_alias in sides:
            table = aliases.get(alias.lower())
            if table not in real_tables:
                continue
            n_rows, n_keys, max_per_key = con.execute(f"""
                SELECT SUM(c), COUNT(*), MAX(c)
                FROM (SELECT COUNT(*) AS c FROM {table} GROUP BY {col})
            """).fetchone()
            if n_rows > n_keys:
                fanned_out = True
                other_table = aliases.get(other_alias.lower(), other_alias)
                warnings.append({
                    "check": "join_fan_out",
                    "severity": "high",
                    "table": table,
                    "column": col,
                    "detail": (f"{table}.{col} is not unique ({n_rows:,} rows for {n_keys:,} keys, "
                               f"up to {max_per_key} rows per key), so each matching row of "
                               f"{other_table} is repeated and the join changes the row count."),
                })

    # CHECK 2: a filter on a nullable column that silently drops unknowns.
    # Why this is a correctness issue, not a style issue:
    # SQL uses three-valued logic. For a row where region is NULL, region != 'West'
    # is neither TRUE nor FALSE, it is UNKNOWN. WHERE keeps only rows whose condition
    # is TRUE, so those rows are removed. No error, no warning, and the result now
    # describes a different population than the analyst meant ("everyone not in the
    # West"). The query runs, but it answers a different question, so the number is
    # wrong, not just untidy. A negative filter that also handles NULL explicitly
    # (OR col IS NULL, or IS DISTINCT FROM) is not flagged.
    where_clauses = re.findall(
        r"\bWHERE\b(.*?)(?=\bGROUP\s+BY\b|\bORDER\s+BY\b|\bHAVING\b|\bLIMIT\b|\)|$)",
        sql, flags=re.IGNORECASE | re.DOTALL,
    )
    for clause in where_clauses:
        for alias, col in re.findall(
            r"(?:(\w+)\.)?(\w+)\s*(?:!=|<>|\bNOT\s+IN\b|\bNOT\s+LIKE\b)", clause, flags=re.IGNORECASE
        ):
            if re.search(rf"\b{col}\s+IS\s+NULL\b", clause, flags=re.IGNORECASE):
                continue
            if alias:
                candidates = [aliases.get(alias.lower())]
            else:
                candidates = [t for t in real_tables if col.lower() in _columns(con, t)]
            for table in candidates:
                if table not in real_tables or col.lower() not in _columns(con, table):
                    continue
                n_null = con.execute(f"SELECT COUNT(*) FROM {table} WHERE {col} IS NULL").fetchone()[0]
                warnings.append({
                    "check": "null_filter_drops_unknowns",
                    "severity": "high" if n_null > 0 else "low",
                    "table": table,
                    "column": col,
                    "detail": (f"Negative filter on {table}.{col}: {n_null:,} rows have NULL there and "
                               f"will be silently excluded. Add OR {col} IS NULL, or use "
                               f"IS DISTINCT FROM, if they should be kept."),
                })

    # CHECK 3: an aggregate computed over a fanned-out result.
    # AVG, SUM and plain COUNT all change when rows are repeated; MIN, MAX and
    # COUNT(DISTINCT ...) do not, so only the first group is flagged.
    if fanned_out and re.search(
        r"\b(?:AVG|SUM)\s*\(|\bCOUNT\s*\(\s*(?!DISTINCT\b)", sql, flags=re.IGNORECASE
    ):
        warnings.append({
            "check": "aggregate_over_fan_out",
            "severity": "high",
            "table": None,
            "column": None,
            "detail": ("An AVG, SUM or COUNT runs over joined rows that repeat a key, so each "
                       "entity is weighted by how many times it was repeated. Aggregate at the "
                       "entity's own grain first, or drop the join."),
        })

    return warnings

In [12]:
tests = [
    ("Phase 1, un-joined avg", f"SELECT AVG(revenue) FROM ({PLAIN_ROWS})"),
    ("Phase 1, joined avg", f"SELECT AVG(revenue) FROM ({JOINED_ROWS})"),
    ("Phase 2, != filter", "SELECT COUNT(*) FROM customers WHERE region != 'West'"),
    ("Phase 2, IS DISTINCT FROM", "SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'"),
]

for label, sql in tests:
    print(label)
    for w in review_query(con, sql):
        print("  ", w["severity"], "|", w["check"], "|", w["detail"])
    print()

Phase 1, un-joined avg

Phase 1, joined avg
   high | join_fan_out | orders.customer_id is not unique (20,490 rows for 3,000 keys, up to 57 rows per key), so each matching row of customers is repeated and the join changes the row count.
   high | aggregate_over_fan_out | An AVG, SUM or COUNT runs over joined rows that repeat a key, so each entity is weighted by how many times it was repeated. Aggregate at the entity's own grain first, or drop the join.

Phase 2, != filter
   high | null_filter_drops_unknowns | Negative filter on customers.region: 450 rows have NULL there and will be silently excluded. Add OR region IS NULL, or use IS DISTINCT FROM, if they should be kept.

Phase 2, IS DISTINCT FROM





The review tool caught the problems it was built to find, flagging the joined Phase 1 query twice, once for the fan-out caused by orders.customer_id not being unique and once for the AVG being computed over those repeated rows, which means it would have caught the inflated number before it reached the board. It also flagged the Phase 2 != query for silently dropping the 450 customers with a NULL region, while correctly staying quiet on both the un-joined query and the IS DISTINCT FROM query.

The tool does have gaps, though, since it finds problems by matching patterns in the SQL text rather than actually understanding the query, so it would miss joins written with USING, queries that use CTEs, and NOT filters placed inside HAVING or a JOIN’s ON clause. It also can’t tell the difference between an intended fan-out and a mistake, so a query that is supposed to count orders per customer would still get flagged, which makes it useful as a first check but not a replacement for someone actually reviewing each warning.

### Task 4.2: The one-page writeup (8 pts)

To the Director,
The correct average revenue for West customers is 814.37 dollars, and the higher number came from a query that joined customers to orders, which counted each customer once for every order they placed. Since customers with more orders also bring in more revenue, they were counted many more times than everyone else and pushed the average up by roughly one and a half times, even though revenue is a customer-level number that should be averaged with one row per customer.

As for the program, a simple comparison makes it look like it adds about 208.64 dollars per customer, but most of that gap exists because the West already had larger accounts, and once account size is accounted for the effect drops to about 21.93 dollars, close to the true 20 dollars. The program does help, just much less than the first comparison suggested. Adding support tickets as a control pushed the estimate down to −30.62 dollars, because the program itself changes how many tickets customers file, so it should not be used as a control.

Before any number goes on a slide, I recommend the team check that the row count still matches the number of customers after every join, check whether a filter is quietly dropping

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* Opus 5.5, 11/1/2026

*Prompt:*

[Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.


------------------------

*Reply:*

import re
import duckdb
import pandas as pd

_SQL_KEYWORDS = {"WHERE", "JOIN", "LEFT", "RIGHT", "INNER", "FULL", "OUTER", "CROSS",
                 "ON", "USING", "GROUP", "ORDER", "HAVING", "LIMIT", "UNION", "AS"}


def _alias_map(sql):
    # Map every alias (o, c) and every bare table name to the real table it points at.
    aliases = {}
    pattern = r"\b(?:FROM|JOIN)\s+([A-Za-z_]\w*)(?:\s+(?:AS\s+)?([A-Za-z_]\w*))?"
    for table, alias in re.findall(pattern, sql, flags=re.IGNORECASE):
        aliases[table.lower()] = table.lower()
        if alias and alias.upper() not in _SQL_KEYWORDS:
            aliases[alias.lower()] = table.lower()
    return aliases


def _columns(con, table):
    # The column names DuckDB has on file for a table; an empty set means it is not a real table.
    rows = con.execute(
        "SELECT column_name FROM information_schema.columns WHERE table_name = ?", [table]
    ).fetchall()
    return {r[0].lower() for r in rows}


def review_query(con, sql):
    warnings = []
    aliases = _alias_map(sql)
    real_tables = {t for t in set(aliases.values()) if _columns(con, t)}
    fanned_out = False

    # CHECK 1: a join that changes the row count.
    # How fan-out is detected without running the full join twice:
    # a join can only multiply rows when the join key repeats on at least one side.
    # So instead of running the join and comparing row counts, each side of every
    # ON a.x = b.y condition is checked on its own table: one GROUP BY on that single
    # table returns the total rows, the number of distinct keys, and the most rows any
    # one key has. If total rows > distinct keys, every matching row on the other side
    # will be repeated, up to max_per_key times. The join itself is never executed.
    for left_alias, left_col, right_alias, right_col in re.findall(
        r"\bON\s+(\w+)\.(\w+)\s*=\s*(\w+)\.(\w+)", sql, flags=re.IGNORECASE
    ):
        sides = [(left_alias, left_col, right_alias), (right_alias, right_col, left_alias)]
        for alias, col, other_alias in sides:
            table = aliases.get(alias.lower())
            if table not in real_tables:
                continue
            n_rows, n_keys, max_per_key = con.execute(f"""
                SELECT SUM(c), COUNT(*), MAX(c)
                FROM (SELECT COUNT(*) AS c FROM {table} GROUP BY {col})
            """).fetchone()
            if n_rows > n_keys:
                fanned_out = True
                other_table = aliases.get(other_alias.lower(), other_alias)
                warnings.append({
                    "check": "join_fan_out",
                    "severity": "high",
                    "table": table,
                    "column": col,
                    "detail": (f"{table}.{col} is not unique ({n_rows:,} rows for {n_keys:,} keys, "
                               f"up to {max_per_key} rows per key), so each matching row of "
                               f"{other_table} is repeated and the join changes the row count."),
                })

    # CHECK 2: a filter on a nullable column that silently drops unknowns.
    # Why this is a correctness issue, not a style issue:
    # SQL uses three-valued logic. For a row where region is NULL, region != 'West'
    # is neither TRUE nor FALSE, it is UNKNOWN. WHERE keeps only rows whose condition
    # is TRUE, so those rows are removed. No error, no warning, and the result now
    # describes a different population than the analyst meant ("everyone not in the
    # West"). The query runs, but it answers a different question, so the number is
    # wrong, not just untidy. A negative filter that also handles NULL explicitly
    # (OR col IS NULL, or IS DISTINCT FROM) is not flagged.
    where_clauses = re.findall(
        r"\bWHERE\b(.*?)(?=\bGROUP\s+BY\b|\bORDER\s+BY\b|\bHAVING\b|\bLIMIT\b|\)|$)",
        sql, flags=re.IGNORECASE | re.DOTALL,
    )
    for clause in where_clauses:
        for alias, col in re.findall(
            r"(?:(\w+)\.)?(\w+)\s*(?:!=|<>|\bNOT\s+IN\b|\bNOT\s+LIKE\b)", clause, flags=re.IGNORECASE
        ):
            if re.search(rf"\b{col}\s+IS\s+NULL\b", clause, flags=re.IGNORECASE):
                continue
            if alias:
                candidates = [aliases.get(alias.lower())]
            else:
                candidates = [t for t in real_tables if col.lower() in _columns(con, t)]
            for table in candidates:
                if table not in real_tables or col.lower() not in _columns(con, table):
                    continue
                n_null = con.execute(f"SELECT COUNT(*) FROM {table} WHERE {col} IS NULL").fetchone()[0]
                warnings.append({
                    "check": "null_filter_drops_unknowns",
                    "severity": "high" if n_null > 0 else "low",
                    "table": table,
                    "column": col,
                    "detail": (f"Negative filter on {table}.{col}: {n_null:,} rows have NULL there and "
                               f"will be silently excluded. Add OR {col} IS NULL, or use "
                               f"IS DISTINCT FROM, if they should be kept."),
                })

    # CHECK 3: an aggregate computed over a fanned-out result.
    # AVG, SUM and plain COUNT all change when rows are repeated; MIN, MAX and
    # COUNT(DISTINCT ...) do not, so only the first group is flagged.
    if fanned_out and re.search(
        r"\b(?:AVG|SUM)\s*\(|\bCOUNT\s*\(\s*(?!DISTINCT\b)", sql, flags=re.IGNORECASE
    ):
        warnings.append({
            "check": "aggregate_over_fan_out",
            "severity": "high",
            "table": None,
            "column": None,
            "detail": ("An AVG, SUM or COUNT runs over joined rows that repeat a key, so each "
                       "entity is weighted by how many times it was repeated. Aggregate at the "
                       "entity's own grain first, or drop the join."),
        })

    return warnings

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.